# Seleção dinâmica de classificadores

Dynamic Classifier Selection (DCS)

***

Vamos precisar primeiramente instalar a biblioteca [Deslib](https://deslib.readthedocs.io/en/latest/index.html)

In [ ]:
!pip install deslib

In [ ]:
!pip install ucimlrepo

In [ ]:
#importando as bibliotecas

import pandas as pd

from sklearn.ensemble import BaggingClassifier
from deslib.static import Oracle, StaticSelection, SingleBest

from deslib.dcs import OLA, LCA
from deslib.des import KNORAU, KNORAE, METADES

from deslib.util.aggregation import majority_voting, average_combiner, minimum_combiner, maximum_combiner, product_combiner

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import warnings
warnings.filterwarnings("ignore")

seed = 42

## Conjunto de dados

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17)

# data (as pandas dataframes)
X = breast_cancer_wisconsin_diagnostic.data.features
y = breast_cancer_wisconsin_diagnostic.data.targets

dataset = pd.concat([X,y], axis=1)
dataset

,radius1,texture1,perimeter1,area1,smoothness1,compactness1,concavity1,concave_points1,symmetry1,fractal_dimension1,...,texture3,perimeter3,area3,smoothness3,compactness3,concavity3,concave_points3,symmetry3,fractal_dimension3,Diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,M
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,M
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,M
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,M
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,M
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,M
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,M
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,M


In [ ]:
#Mapeando os valores da classe para inteiro (para fins de visualização)
dataset["Diagnosis"] = pd.factorize(dataset["Diagnosis"])[0]
# M = 0
# B = 1

In [ ]:
X = dataset.drop([dataset.columns[30]], axis = 1)
y = dataset[dataset.columns[30]]

# separando o conjunto de dados em treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)

# separando o conjunto de dados em treinamento e DSEL para as técnicas de DS
X_train, X_dsel, y_train, y_dsel = train_test_split(X_train, y_train, test_size=0.5, stratify=y_train, random_state=seed)

# Seleção estática

Na seleção estática, a seleção dos classificadores é realizada na fase de treinamento.

O **Oracle** é um método abstrato que representa um esquema de seleção de classificador ideal.Ele sempre seleciona o classificador que previu o rótulo correto, para a amostra de consulta fornecida, se tal classificador existir.

In [ ]:
# Pool de modelos
pool_classifiers = BaggingClassifier(n_estimators=2, random_state=seed)
pool_classifiers.fit(X_train, y_train)

print(pool_classifiers.estimators_)

[DecisionTreeClassifier(random_state=1952926171), DecisionTreeClassifier(random_state=1761383086)]


In [ ]:
static_result = []
for n in [10, 20, 30]:
  pool_classifiers = BaggingClassifier(n_estimators=n, random_state=seed)
  pool_classifiers.fit(X_train, y_train)

  static_model = StaticSelection(pool_classifiers, random_state=seed)
  single_model = SingleBest(pool_classifiers, random_state=seed)

  static_model.fit(X_dsel, y_dsel)
  single_model.fit(X_dsel, y_dsel)

  static_pred = static_model.predict(X_test)
  single_pred = static_model.predict(X_test)

  majority_pred = majority_voting(pool_classifiers, X_test)
  average_pred = average_combiner(pool_classifiers, X_test)
  product_pred = product_combiner(pool_classifiers, X_test)
  maximum_pred = maximum_combiner(pool_classifiers, X_test)
  minimum_pred = minimum_combiner(pool_classifiers, X_test)

  oracle = Oracle(pool_classifiers).fit(X_train, y_train)
  oracle_pred = oracle.predict(X_test, y_test)

  #calc f1-score for each model
  scores = [n,
            f1_score(y_test, static_pred),
            f1_score(y_test, single_pred),
            f1_score(y_test, majority_pred),
            f1_score(y_test, average_pred),
            f1_score(y_test, product_pred),
            f1_score(y_test, maximum_pred),
            f1_score(y_test, minimum_pred),
            f1_score(y_test, oracle_pred)]

  static_result.append(pd.DataFrame([scores], columns=['size_pool',
                                                       'Static',
                                                       'Single',
                                                       'Majority',
                                                       'Average',
                                                       'Product',
                                                       'Max',
                                                       'Min',
                                                       'Oracle']))

static_result = pd.concat(static_result).reset_index(drop=True).round(3)
static_result

,size_pool,Static,Single,Majority,Average,Product,Max,Min,Oracle
0,10,0.946,0.946,0.936,0.936,0.890,0.890,0.890,0.982
1,20,0.945,0.945,0.941,0.941,0.881,0.881,0.881,0.995
2,30,0.950,0.950,0.945,0.945,0.856,0.856,0.856,1.000


### AutoGluon

Ferramenta de AutoML que permite testar diversas máquinas de aprendizagem de forma rápida e usando poucas linhas de código.

In [ ]:
!pip install autogluon

In [ ]:
df_train, df_test = train_test_split(dataset, test_size=0.3, random_state=seed, stratify=y)

In [97]:
from autogluon.tabular import TabularPredictor

# Treina o modelo
predictor = TabularPredictor(label='Diagnosis').fit(df_train)

# Realiza as predições
predictions = predictor.predict(df_test)
print("Predictions:")
print(predictions)

# You can also get a summary of the trained models and their performance on a validation set
leaderboard = predictor.leaderboard(df_test, silent=True)
print("\nLeaderboard of trained models:")
print(leaderboard)

No path specified. Models will be saved in: "AutogluonModels/ag-20250815_161536"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.11.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sun Mar 30 16:01:29 UTC 2025
CPU Count:          2
Memory Avail:       10.71 GB / 12.67 GB (84.5%)
Disk Space Avail:   63.94 GB / 107.72 GB (59.4%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme' : New in v1.4: Massively better than 'best' on datasets <30000 samples by using new models meta-learned on https://tabarena.ai: TabPFNv2, TabICL, Mitra, and TabM. Absolute best accuracy. Requires a GPU. Recommended 64 GB CPU memory and 32+ GB GPU me

Predictions:
14     0
150    1
275    1
83     0
86     1
      ..
37     1
358    1
363    0
171    0
284    1
Name: Diagnosis, Length: 171, dtype: int64

Leaderboard of trained models:
                  model  score_test  score_val eval_metric  pred_time_test  \
0        NeuralNetTorch    0.964912     0.9750    accuracy        0.020495   
1              CatBoost    0.959064     0.9750    accuracy        0.004348   
2       NeuralNetFastAI    0.953216     0.9875    accuracy        0.013558   
3   WeightedEnsemble_L2    0.953216     0.9875    accuracy        0.015563   
4        ExtraTreesGini    0.953216     0.9625    accuracy        0.098517   
5        ExtraTreesEntr    0.953216     0.9625    accuracy        0.104317   
6         LightGBMLarge    0.947368     0.9750    accuracy        0.004859   
7      RandomForestEntr    0.947368     0.9625    accuracy        0.097983   
8      RandomForestGini    0.947368     0.9625    accuracy        0.099773   
9              LightGBM    0.9356

Existem outras ferramentas de AutoML com abordagens Semelhantes...

Ver:

*   AutoSklearn [Link](https://automl.github.io/auto-sklearn/master/)
*   TPOT [Link](https://github.com/EpistasisLab/tpot)



# Seleção dinâmica

A seleção dos classificadores é realizada para cada nova amostra de teste na fase de classificação.

Relembrando algumas das possibilidades de escolha do classificador dentro da região de competência:

OLA (Overall Local Accuracy): Escolhe o classificador com maior acurácia média dentro de uma determinada região de competência.

LCA (Local Class Accuracy): Escolhe o classificador com maior acurácia para uma classe específica dentro de uma determinada região de competência. Ignora os resultados de outras classes.

KNORAE (k-Nearest Oracle-Eliminate): Seleciona os melhores classificadores dentro da região de competência. Caso múltiplos sejam escolhidos utiliza voto majoritário para definir a classe.

KNORAU (k-Nearest Oracle Union): Seleciona todos os classificadores que classificaram corretamente ao menos 1 exemplo dentro da região de competência. A escolha da classe é feita através de voto.
A quantidade de votos por classificador é dada pela quantidade de instâncias classificadas corretamente.

Vamos avaliar diferentes abordagens:

1 - Seleção dinâmica de classificador

2 - Seleção dinâmica de ensemble

[META-DES](https://www.sciencedirect.com/science/article/pii/S0031320314004919): Framework de seleção de ensemble dinâmico. O META-DES usa meta-aprendizagem para fazer a seleção. São utilizados cinco   conjuntos distintos de metacaracterísticas, cada uma correspondendo a um critério diferente para medir o nível de competência de um classificador para a classificação de amostras de entrada.

In [ ]:
dynamic_result = []
for n in [10, 20, 30]:
  pool_classifiers = BaggingClassifier(n_estimators=n, random_state=seed)
  pool_classifiers.fit(X_train, y_train)

  # Dynamic classifier selection
  ola = OLA(pool_classifiers, random_state=seed)
  lca = LCA(pool_classifiers, random_state=seed)

  # Dynamic ensemble selection
  kne = KNORAE(pool_classifiers, random_state=seed)
  knu = KNORAU(pool_classifiers, random_state=seed)

  #meta-des
  meta = METADES(pool_classifiers, random_state=seed)


  ola.fit(X_dsel, y_dsel)
  lca.fit(X_dsel, y_dsel)
  kne.fit(X_dsel, y_dsel)
  knu.fit(X_dsel, y_dsel)
  meta.fit(X_dsel, y_dsel)

  ola_pred = ola.predict(X_test)
  lca_pred = lca.predict(X_test)
  kne_pred = kne.predict(X_test)
  knu_pred = knu.predict(X_test)
  meta_pred = meta.predict(X_test)

  oracle = Oracle(pool_classifiers).fit(X_train, y_train)
  oracle_pred = oracle.predict(X_test, y_test)

  scores = [n,
            f1_score(y_test, ola_pred),
            f1_score(y_test, lca_pred),
            f1_score(y_test, kne_pred),
            f1_score(y_test, knu_pred),
            f1_score(y_test, meta_pred),
            f1_score(y_test, oracle_pred)]

  dynamic_result.append(pd.DataFrame([scores], columns=['size_pool', 'OLA', 'LCA', 'KNORAE', 'KNORAU','META-DES', 'Oracle']))

dynamic_result = pd.concat(dynamic_result).reset_index(drop=True).round(3)
dynamic_result

,size_pool,OLA,LCA,KNORAE,KNORAU,META-DES,Oracle
0,10,0.950,0.942,0.938,0.945,0.941,0.982
1,20,0.963,0.942,0.959,0.945,0.950,0.995
2,30,0.963,0.942,0.968,0.950,0.945,1.000


### E se quisermos usar um pool heterogênio?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [ ]:
model_svc = SVC(probability=True, gamma='auto',
                random_state=seed).fit(X_train, y_train)
model_tree = DecisionTreeClassifier(random_state=seed,
                                    max_depth=10).fit(X_train, y_train)
model_knn = KNeighborsClassifier(n_neighbors=7).fit(X_train, y_train)

pool_classifiers = [model_svc, model_tree, model_knn]

ola = OLA(pool_classifiers, random_state=seed)

ola.fit(X_dsel, y_dsel)

ola_pred = ola.predict(X_test)

print('F-score do OLA: %.3f' % f1_score(y_test, ola_pred))

F-score do OLA: 0.933


# Outro Dataset (Wine Quality)

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
wine_quality = fetch_ucirepo(id=186)

# data (as pandas dataframes)
X = wine_quality.data.features
y = wine_quality.data.targets


dataset = pd.concat([X,y], axis=1)
dataset

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
...,...,...,...,...,...,...,...,...,...,...,...,...
6492,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
6493,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
6494,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
6495,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [ ]:
#Mapeando atributo qualidade para binário de 0 a 6 => 0 e de 6 a 10 => 1
dataset['quality'] = dataset['quality'].map({1:0,2:0,3:0,4:0,5:0,6:1,7:1,8:1,9:1})
dataset['quality'].value_counts()

,count
quality,
1,4113
0,2384


In [ ]:
X = dataset.drop([dataset.columns[-1]], axis = 1)
y = dataset[dataset.columns[-1]]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed)

# separando o conjunto de dados em treinamento e DSEL para as técnicas de DS
X_train, X_dsel, y_train, y_dsel = train_test_split(X_train, y_train, test_size=0.5, stratify=y_train, random_state=seed)

X_train.shape, X_test.shape
y_train.value_counts()
y_test.value_counts()

,count
quality,
1,1259
0,691


In [ ]:
X_train

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol
3989,7.0,0.28,0.26,1.7,0.042,34.0,130.0,0.99250,3.43,0.50,10.7
4335,6.5,0.17,0.31,1.5,0.041,34.0,121.0,0.99092,3.06,0.46,10.5
4728,6.1,0.21,0.38,1.5,0.039,37.0,122.0,0.98972,3.20,0.43,12.0
6284,7.2,0.26,0.32,10.4,0.062,23.0,114.0,0.99660,3.23,0.49,10.5
1649,7.2,0.19,0.31,1.6,0.062,31.0,173.0,0.99170,3.35,0.44,11.7
...,...,...,...,...,...,...,...,...,...,...,...
2605,5.8,0.36,0.26,3.3,0.038,40.0,153.0,0.99110,3.34,0.55,11.3
5725,6.3,0.18,0.22,5.6,0.047,45.0,147.0,0.99383,3.09,0.54,10.0
4021,6.2,0.44,0.18,7.7,0.096,28.0,210.0,0.99771,3.56,0.72,9.2
2038,6.2,0.35,0.04,1.2,0.060,23.0,108.0,0.99340,3.26,0.54,9.2


In [ ]:
X_dsel

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol
4419,6.7,0.325,0.82,1.2,0.152,49.0,120.0,0.99312,2.99,0.38,9.2
5681,5.9,0.230,0.28,8.6,0.046,37.0,142.0,0.99432,3.23,0.53,10.6
2547,6.8,0.705,0.25,3.2,0.048,10.0,57.0,0.99600,3.36,0.52,9.5
4869,6.4,0.150,0.29,1.8,0.044,21.0,115.0,0.99166,3.10,0.38,10.2
5638,6.1,1.100,0.16,4.4,0.033,8.0,109.0,0.99058,3.35,0.47,12.4
...,...,...,...,...,...,...,...,...,...,...,...
1134,8.5,0.280,0.35,1.7,0.061,6.0,15.0,0.99524,3.30,0.74,11.8
5925,7.4,0.380,0.34,8.3,0.052,44.0,168.0,0.99627,3.11,0.52,9.2
1777,6.0,0.670,0.07,1.2,0.060,9.0,108.0,0.99310,3.11,0.35,8.7
4899,7.8,0.250,0.37,1.0,0.043,10.0,80.0,0.99128,3.08,0.38,11.4


In [ ]:
X_test

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol
3103,7.0,0.17,0.74,12.80,0.045,24.0,126.0,0.99420,3.26,0.38,12.2
1419,7.7,0.64,0.21,2.20,0.077,32.0,133.0,0.99560,3.27,0.45,9.9
4761,6.8,0.39,0.34,7.40,0.020,38.0,133.0,0.99212,3.18,0.44,12.0
4690,6.3,0.28,0.47,11.20,0.040,61.0,183.0,0.99592,3.12,0.51,9.5
4032,7.4,0.35,0.20,13.90,0.054,63.0,229.0,0.99888,3.11,0.50,8.9
...,...,...,...,...,...,...,...,...,...,...,...
4294,5.7,0.23,0.28,9.65,0.025,26.0,121.0,0.99250,3.28,0.38,11.3
3757,7.4,0.18,0.27,1.30,0.048,26.0,105.0,0.99400,3.52,0.66,10.6
5954,6.4,0.31,0.28,2.50,0.039,34.0,137.0,0.98946,3.22,0.38,12.7
4418,6.0,0.21,0.34,2.00,0.042,63.0,123.0,0.99052,3.44,0.42,11.4


In [ ]:
# Pool de modelos
pool_classifiers = BaggingClassifier(n_estimators=2, random_state=seed)
pool_classifiers.fit(X_train, y_train)

print(pool_classifiers.estimators_)

[DecisionTreeClassifier(random_state=1952926171), DecisionTreeClassifier(random_state=1761383086)]


In [ ]:
static_result = []
for n in [10, 20, 30]:
  pool_classifiers = BaggingClassifier(n_estimators=n, random_state=seed)
  pool_classifiers.fit(X_train, y_train)

  static_model = StaticSelection(pool_classifiers, random_state=seed)
  single_model = SingleBest(pool_classifiers, random_state=seed)

  static_model.fit(X_dsel, y_dsel)
  single_model.fit(X_dsel, y_dsel)

  static_pred = static_model.predict(X_test)
  single_pred = static_model.predict(X_test)

  majority_pred = majority_voting(pool_classifiers, X_test)
  average_pred = average_combiner(pool_classifiers, X_test)
  product_pred = product_combiner(pool_classifiers, X_test)
  maximum_pred = maximum_combiner(pool_classifiers, X_test)
  minimum_pred = minimum_combiner(pool_classifiers, X_test)

  oracle = Oracle(pool_classifiers, random_state=seed).fit(X_train, y_train)
  oracle_pred = oracle.predict(X_test, y_test)

  #calc f1-score for each model
  scores = [n,
            f1_score(y_test, static_pred),
            f1_score(y_test, single_pred),
            f1_score(y_test, majority_pred),
            f1_score(y_test, average_pred),
            f1_score(y_test, product_pred),
            f1_score(y_test, maximum_pred),
            f1_score(y_test, minimum_pred),
            f1_score(y_test, oracle_pred)]

  static_result.append(pd.DataFrame([scores], columns=['size_pool',
                                                       'Static',
                                                       'Single',
                                                       'Majority',
                                                       'Average',
                                                       'Product',
                                                       'Max',
                                                       'Min',
                                                       'Oracle']))

static_result = pd.concat(static_result).reset_index(drop=True).round(3)
static_result

,size_pool,Static,Single,Majority,Average,Product,Max,Min,Oracle
0,10,0.811,0.811,0.803,0.803,0.464,0.464,0.464,0.984
1,20,0.803,0.803,0.810,0.810,0.319,0.319,0.319,0.994
2,30,0.820,0.820,0.814,0.814,0.243,0.243,0.243,0.996


In [ ]:
dynamic_result = []
for n in [10, 20, 30]:
  pool_classifiers = BaggingClassifier(n_estimators=n, random_state=seed)
  pool_classifiers.fit(X_train, y_train)

  # Dynamic classifier selection
  ola = OLA(pool_classifiers, random_state=seed)
  lca = LCA(pool_classifiers, random_state=seed)

  # Dynamic ensemble selection
  kne = KNORAE(pool_classifiers, random_state=seed)
  knu = KNORAU(pool_classifiers, random_state=seed)

  #meta-des
  meta = METADES(pool_classifiers, random_state=seed)


  ola.fit(X_dsel, y_dsel)
  lca.fit(X_dsel, y_dsel)
  kne.fit(X_dsel, y_dsel)
  knu.fit(X_dsel, y_dsel)
  meta.fit(X_dsel, y_dsel)

  ola_pred = ola.predict(X_test)
  lca_pred = lca.predict(X_test)
  kne_pred = kne.predict(X_test)
  knu_pred = knu.predict(X_test)
  meta_pred = meta.predict(X_test)

  oracle = Oracle(pool_classifiers, random_state=seed).fit(X_train, y_train)
  oracle_pred = oracle.predict(X_test, y_test)

  scores = [n,
            f1_score(y_test, ola_pred),
            f1_score(y_test, lca_pred),
            f1_score(y_test, kne_pred),
            f1_score(y_test, knu_pred),
            f1_score(y_test, meta_pred),
            f1_score(y_test, oracle_pred)]

  dynamic_result.append(pd.DataFrame([scores], columns=['size_pool', 'OLA', 'LCA', 'KNORAE', 'KNORAU','META-DES', 'Oracle']))

dynamic_result = pd.concat(dynamic_result).reset_index(drop=True).round(3)
dynamic_result

,size_pool,OLA,LCA,KNORAE,KNORAU,META-DES,Oracle
0,10,0.799,0.771,0.806,0.823,0.807,0.984
1,20,0.797,0.762,0.799,0.824,0.827,0.994
2,30,0.805,0.758,0.805,0.826,0.834,0.996
